# 多模型输出稳定性练习

目标：在不同模型下，让输出尽量稳定（结构一致、字段完整、措辞波动可控）。


In [6]:
%pip install -q langchain langchain-openai
# 如果你是本地服务（如 LM Studio / Ollama 网关），保持 base_url 可用

You should consider upgrading via the '/Users/ludan/Downloads/git-repo/chanon-data-lab/.docs/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
# 导入：链路编排 + 提示模板 + 聊天模型
from langchain.chains import LLMChain
from langchain import PromptTemplate
from langchain_openai import ChatOpenAI
import json
from collections import defaultdict

In [ ]:
# 1) 模型层：准备多个模型配置（可按你本地实际模型名修改）
LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"

MODEL_SETUPS = [
    {"name": "qwen2.5-7b-instruct", "temperature": 0},
    {"name": "qwen2.5-7b-instruct", "temperature": 1},
    {"name": "llama3.1-8b-instruct", "temperature": 0},
    {"name": "mistral-7b-instruct", "temperature": 0},
]

def build_llm(model_name: str, temperature: float = 0):
    return ChatOpenAI(
        base_url=LM_STUDIO_BASE,
        api_key="lm-studio",
        model=model_name,
        temperature=temperature,
    )

In [9]:
# 2) 提示层：强约束输出结构，减少模型自由度
# 练习重点：即便换模型，也要尽量输出同样的 JSON 结构。
template = """
你是一个严格的信息抽取器。请严格返回 JSON，不要返回任何额外文本。

任务：根据用户问题生成稳定答复。

硬性要求：
1) 只能输出一个 JSON 对象。
2) 字段必须且仅能是：answer, confidence, rationale
3) answer: 不超过30字
4) confidence: 0 到 1 之间，保留2位小数
5) rationale: 不超过20字
6) 如果无法判断，也必须按同样字段返回

用户问题：{question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
)

In [10]:
# 3) 编排层：给每个模型各建一条链
chains = {}
for setup in MODEL_SETUPS:
    llm = build_llm(setup["name"], setup["temperature"])
    chains[setup["name"]] = LLMChain(prompt=prompt, llm=llm, verbose=False)

print("已初始化模型链：", list(chains.keys()))

已初始化模型链： ['qwen2.5-7b-instruct', 'llama3.1-8b-instruct', 'mistral-7b-instruct']


In [11]:
# 4) 调用与评估：同一问题多次调用，比较不同模型的“格式稳定性”

questions = [
    "明天北京下雨概率高吗？",
    "咖啡晚上喝会影响睡眠吗？",
    "跑步前要不要吃东西？",
]

def parse_json_safely(text: str):
    text = text.strip()
    try:
        return json.loads(text), True
    except Exception:
        return {"raw": text}, False

def check_schema(data: dict):
    expected = {"answer", "confidence", "rationale"}
    return set(data.keys()) == expected

results = defaultdict(list)

for model_name, chain in chains.items():
    for q in questions:
        # 每个问题重复 3 次，看模型在同温度下是否稳定
        for i in range(3):
            raw = chain.predict(question=q)
            data, is_json = parse_json_safely(raw)
            schema_ok = is_json and check_schema(data)
            results[model_name].append(
                {
                    "question": q,
                    "run": i + 1,
                    "raw": raw,
                    "is_json": is_json,
                    "schema_ok": schema_ok,
                    "data": data,
                }
            )

# 打印每个模型的稳定性统计
for model_name, rows in results.items():
    total = len(rows)
    json_ok = sum(r["is_json"] for r in rows)
    schema_ok = sum(r["schema_ok"] for r in rows)
    print(f"\n===== {model_name} =====")
    print(f"JSON 合法率: {json_ok}/{total} = {json_ok/total:.0%}")
    print(f"Schema 稳定率: {schema_ok}/{total} = {schema_ok/total:.0%}")

    # 展示一个样本
    sample = rows[0]
    print("样本输出:")
    print(sample["raw"])

print("\n练习建议：把 temperature 改为 0.2 / 0.7，再观察稳定率变化。")


===== qwen2.5-7b-instruct =====
JSON 合法率: 9/9 = 100%
Schema 稳定率: 9/9 = 100%
样本输出:
{"answer": "无法确定", "confidence": 0.50, "rationale": "需查天气预报"}

===== llama3.1-8b-instruct =====
JSON 合法率: 9/9 = 100%
Schema 稳定率: 9/9 = 100%
样本输出:
{"answer": "无法确定", "confidence": 0.50, "rationale": "需查天气预报"}

===== mistral-7b-instruct =====
JSON 合法率: 9/9 = 100%
Schema 稳定率: 9/9 = 100%
样本输出:
{"answer": "无法确定", "confidence": 0.50, "rationale": "需查天气预报"}

练习建议：把 temperature 改为 0.2 / 0.7，再观察稳定率变化。


## 进阶任务

1. 增加一个 `format_instructions` 版本的提示词（例如用分隔符包裹 JSON），对比稳定率。  
2. 在相同模型上比较 `temperature=0` 和 `temperature=0.7`。  
3. 统计字段长度分布，观察 `answer` 是否超出 30 字。